In [2]:
import gurobipy as gp
from gurobipy import GRB
from matpowercaseframes import CaseFrames
import joblib
import numpy as np
from tqdm import tqdm
import pathlib, re

from sys import stderr
from numpy import zeros, arange, isscalar, dot, ix_, ones, r_, pi, flatnonzero as find
from scipy.sparse import csr_matrix as sparse
from pypower.idx_bus import BUS_TYPE, REF, BUS_I
from pypower.idx_brch import F_BUS, T_BUS, BR_X, TAP, SHIFT, BR_STATUS
from numpy.linalg import solve

In [3]:
def makeBdc(baseMVA, bus, branch):
    """Builds the B matrices and phase shift injections for DC power flow.

    Returns the B matrices and phase shift injection vectors needed for a
    DC power flow.
    The bus real power injections are related to bus voltage angles by::
        P = Bbus * Va + PBusinj
    The real power flows at the from end the lines are related to the bus
    voltage angles by::
        Pf = Bf * Va + Pfinj
    Does appropriate conversions to p.u.
    @see: L{dcpf}
    @author: Carlos E. Murillo-Sanchez (PSERC Cornell & Universidad
    Autonoma de Manizales)
    @author: Ray Zimmerman (PSERC Cornell)
    """
    ## constants
    nb = bus.shape[0]          ## number of buses
    nl = branch.shape[0]       ## number of lines

    ## check that bus numbers are equal to indices to bus (one set of bus nums)
    if any(bus[:, BUS_I]-1 != list(range(nb))):
        stderr.write('makeBdc: buses must be numbered consecutively in '
                     'bus matrix\n')

    ## for each branch, compute the elements of the branch B matrix and the phase
    ## shift "quiescent" injections, where
    ##
    ##      | Pf |   | Bff  Bft |   | Vaf |   | Pfinj |
    ##      |    | = |          | * |     | + |       |
    ##      | Pt |   | Btf  Btt |   | Vat |   | Ptinj |
    ##
    stat = branch[:, BR_STATUS]               ## ones at in-service branches
    b = stat / branch[:, BR_X]                ## series susceptance
    tap = ones(nl)                            ## default tap ratio = 1
    i = find(branch[:, TAP])               ## indices of non-zero tap ratios
    tap[i] = branch[i, TAP]                   ## assign non-zero tap ratios
    b = b / tap

    ## build connection matrix Cft = Cf - Ct for line and from - to buses
    f = branch[:, F_BUS] -1                           ## list of "from" buses
    t = branch[:, T_BUS] -1                          ## list of "to" buses
    i = r_[range(nl), range(nl)]                   ## double set of row indices
    ## connection matrix
    Cft = sparse((r_[ones(nl), -ones(nl)], (i, r_[f, t])), (nl, nb))

    ## build Bf such that Bf * Va is the vector of real branch powers injected
    ## at each branch's "from" bus
    Bf = sparse((r_[b, -b], (i, r_[f, t])), shape = (nl, nb))## = spdiags(b, 0, nl, nl) * Cft

    ## build Bbus
    Bbus = Cft.T * Bf

    ## build phase shift injection vectors
    Pfinj = b * (-branch[:, SHIFT] * pi / 180)  ## injected at the from bus ...
    # Ptinj = -Pfinj                            ## and extracted at the to bus
    Pbusinj = Cft.T * Pfinj                ## Pbusinj = Cf * Pfinj + Ct * Ptinj

    return Bbus, Bf, Pbusinj, Pfinj

def makePTDF(baseMVA, bus, branch, slack=None):
    """Builds the DC PTDF matrix for a given choice of slack.

    Returns the DC PTDF matrix for a given choice of slack. The matrix is
    C{nbr x nb}, where C{nbr} is the number of branches and C{nb} is the
    number of buses. The C{slack} can be a scalar (single slack bus) or an
    C{nb x 1} column vector of weights specifying the proportion of the
    slack taken up at each bus. If the C{slack} is not specified the
    reference bus is used by default.

    For convenience, C{slack} can also be an C{nb x nb} matrix, where each
    column specifies how the slack should be handled for injections
    at that bus.

    @see: L{makeLODF}

    @author: Ray Zimmerman (PSERC Cornell)
    """
    ## use reference bus for slack by default
    if slack is None:
        slack = find(bus[:, BUS_TYPE] == REF)
        slack = slack[0]

    ## set the slack bus to be used to compute initial PTDF
    if isscalar(slack):
        slack_bus = slack
    else:
        slack_bus = 0      ## use bus 1 for temp slack bus

    nb = bus.shape[0]
    nbr = branch.shape[0]
    noref = arange(1, nb)      ## use bus 1 for voltage angle reference
    noslack = find(arange(nb) != slack_bus)

    ## check that bus numbers are equal to indices to bus (one set of bus numbers)
    if any(bus[:, BUS_I]-1 != arange(nb)):
        stderr.write('makePTDF: buses must be numbered consecutively')

    ## compute PTDF for single slack_bus
    Bbus, Bf, _, _ = makeBdc(baseMVA, bus, branch)
    Bbus, Bf = Bbus.todense(), Bf.todense()
    H = zeros((nbr, nb))
    H[:, noslack] = solve( Bbus[ix_(noslack, noref)].T, Bf[:, noref].T ).T
    #             = Bf[:, noref] * inv(Bbus[ix_(noslack, noref)])

    ## distribute slack, if requested
    if not isscalar(slack):
        if len(slack.shape) == 1:  ## slack is a vector of weights
            slack = slack / sum(slack)   ## normalize weights

            ## conceptually, we want to do ...
            ##    H = H * (eye(nb, nb) - slack * ones((1, nb)))
            ## ... we just do it more efficiently
            v = dot(H, slack)
            for k in range(nb):
                H[:, k] = H[:, k] - v
        else:
            H = dot(H, slack)

    return H                

def create_scenario_multivariate(case, model, N, sigma_scaling = 0.03):
    buses = case.bus.values
    buses_cols = {col:num for num,col in enumerate(case.bus.columns.values)}
    base_demand = buses[:,buses_cols['PD']] / case.baseMVA
    sigma = (sigma_scaling * base_demand)
    mean = np.zeros(len(base_demand))
    covariance_matrix = np.diag(sigma**2)
    omega = np.random.multivariate_normal(mean, covariance_matrix, N)
    return omega

def update_injection_constraints(case, model, omega_bound):
    omega = case.omega
    omega.LB = omega_bound
    omega.UB = omega_bound
    model.update()  # Update the model to reflect these changes


In [9]:
case_name = 'pglib_opf_case3_lmbd'
case_path = f'..\\pglib-opf-21.07\\{case_name}.m'
case = CaseFrames(case_path)
bus_to_idx = {bus: i+1 for i, bus in enumerate(case.bus.BUS_I.values)}
bus_idx = [bus_to_idx[bus] for bus in case.bus.BUS_I.values]
case.bus.BUS_I = case.bus.BUS_I.replace(bus_to_idx) # rename the bus for making PTDF
case.gen.GEN_BUS = case.gen.GEN_BUS.replace(bus_to_idx)
case.branch.F_BUS = case.branch.F_BUS.replace(bus_to_idx)
case.branch.T_BUS = case.branch.T_BUS.replace(bus_to_idx)
pmax = case.gen.PMAX.values/case.baseMVA
pmin = case.gen.PMIN.values/case.baseMVA
zero_gen_idx = []
for num,i in enumerate(pmax):
    if i == 0 and pmin[num] == 0:
        zero_gen_idx.append(num+1)
case.gen.drop(index=zero_gen_idx, inplace=True) # drop
case.gencost.drop(index=zero_gen_idx, inplace=True) # drop
pmax = np.delete(pmax, np.array(zero_gen_idx).astype(np.int32)-1)
pmin = np.delete(pmin, np.array(zero_gen_idx).astype(np.int32)-1)
case.M = makePTDF(case.baseMVA, case.bus.values, case.branch.values, slack=None)
nbus = case.bus.shape[0]
ngen = case.gen.shape[0]
nbranch = case.branch.shape[0]
case.H = np.zeros((nbus,ngen))
for gen,bus in enumerate(case.gen.GEN_BUS.values):
    case.H[int(bus)-1][gen] = 1
fmax = case.branch.RATE_A.values/case.baseMVA
d0 = case.bus.PD.values/case.baseMVA
c2 = case.gencost.C2.values * case.baseMVA**2
c1 = case.gencost.C1.values * case.baseMVA
c0 = case.gencost.C0.values
c = c2 + c1 + c0
c = np.hstack([c,np.zeros(ngen*2+nbranch*2)])

In [11]:
# primal problem with explicit slacks
def lpformulator_dc_body_primal(case, model):
    _add_dc_gen_bus_variables(case, model)				
    set_gencost_objective_primal(case, model)			# (1a✓) 
    _add_dc_bus_balance_constraints(case, model)		# (1b✓)
    _add_generator_limit_constraints(case, model)       # (1c✓)
    _add_branch_limit_constraints(case, model)          # (1d✓)

def _add_dc_gen_bus_variables(case, model):  #(1b)
    gens = case.gen.values
    branches = case.branch.values
    buses = case.bus.values
    p = model.addMVar(shape=len(gens),lb=0, ub=GRB.INFINITY , name='p')
    r_u = model.addMVar(shape=len(gens),lb=0, ub=GRB.INFINITY , name='r_u')
    q_u = model.addMVar(shape=len(branches),lb=0, ub=GRB.INFINITY , name='q_u')
    q_l = model.addMVar(shape=len(branches),lb=0, ub=GRB.INFINITY , name='q_l')
    omega = model.addMVar(shape=len(buses), lb=-GRB.INFINITY, ub=GRB.INFINITY, name='omega')
    case.p = p
    case.r_u = r_u
    case.q_u = q_u
    case.q_l = q_l
    case.omega = omega

def _add_dc_bus_balance_constraints(case, model): # (1b)
    p = case.p
    r_u = case.r_u
    q_u = case.q_u
    q_l = case.q_l
    omega = case.omega
    buses = case.bus.values
    buses_cols = {col:num for num,col in enumerate(case.bus.columns.values)}
    Pd = buses[:,buses_cols['PD']]/case.baseMVA
    pmin = case.gen.PMIN.values / case.baseMVA
    model.addConstr(p.sum() 
                    + np.zeros(r_u.shape[0])@r_u 
                    + np.zeros(q_u.shape[0])@q_u 
                    + np.zeros(q_l.shape[0])@q_l
                      == Pd.sum() - pmin.sum()  + omega.sum(), name="(4b)")

def _add_generator_limit_constraints(case, model):
    gens = case.gen.values
    p = case.p
    r_u = case.r_u
    q_u = case.q_u
    q_l = case.q_l    
    gens_cols = {col:num for num,col in enumerate(case.gen.columns.values)}    
    Pmax = gens[:,gens_cols['PMAX']]/case.baseMVA
    Pmin = gens[:,gens_cols['PMIN']]/case.baseMVA    
    status = gens[:,gens_cols['GEN_STATUS']]
    model.addConstr(p + r_u 
                    + np.zeros(q_u.shape[0])@q_u 
                    + np.zeros(q_l.shape[0])@q_l
                     == (Pmax-Pmin) * status, name="(4c)")

def _add_branch_limit_constraints(case, model): #(1d)
    branches = case.branch.values
    branches_cols = {col:num for num,col in enumerate(case.branch.columns.values)}
    buses = case.bus.values
    buses_cols = {col:num for num,col in enumerate(case.bus.columns.values)}    
    M, H = case.M, case.H
    p = case.p
    r_u = case.r_u
    q_u = case.q_u
    q_l = case.q_l
    omega = case.omega           
    d = buses[:,buses_cols['PD']]/case.baseMVA
    limit = branches[:,branches_cols['RATE_A']]/case.baseMVA
    pmin = case.gen.PMIN.values / case.baseMVA
    # MdwHp = case.M @ (d + omega - case.H@pmin)
    model.addConstr(M@H@p 
                    + np.zeros(r_u.shape[0])@r_u 
                    + q_u 
                    + np.zeros(q_l.shape[0])@q_l
                    == limit + M@(d+omega) - M@H@pmin, name="(4d)")    
    model.addConstr(-M@H@p 
                    +np.zeros(r_u.shape[0])@r_u 
                    + np.zeros(q_u.shape[0])@q_u 
                    + q_l
                    == limit - M@(d+omega) + M@H@pmin, name="(4e)")

def set_gencost_objective_primal(case, model): # (1a)
    p = case.p
    pmin = case.gen.PMIN.values / case.baseMVA
    costvector = case.gencost[['C2', 'C1', 'C0']].values
    objective_quadratic = (costvector[:, -3]*case.baseMVA**2 ) @ (p + pmin)
    objective_linear = (costvector[:, -2]*case.baseMVA) @ (p + pmin)
    objective_constant = costvector[:, -1] @ (p + pmin)
    model.setObjective(
        objective_quadratic + objective_constant + objective_linear , sense=GRB.MINIMIZE
    )    

model_primal = gp.Model(f"DC_Formulation_Model_primal_{case_name}")
# model_primal.params.OutputFlag = 0
lpformulator_dc_body_primal(case, model_primal)
omega_0 = create_scenario_multivariate(case, model_primal, 1, sigma_scaling=0)
update_injection_constraints(case, model_primal, omega_0[0])
model_primal.update()
model_primal.optimize()

Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: 12th Gen Intel(R) Core(TM) i7-1255U, instruction set [SSE2|AVX|AVX2]
Thread count: 10 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 9 rows, 13 columns and 33 nonzeros (Min)
Model fingerprint: 0xf0428f95
Model has 2 linear objective coefficients
Coefficient statistics:
  Matrix range     [3e-01, 1e+00]
  Objective range  [1e+03, 2e+03]
  Bounds range     [0e+00, 0e+00]
  RHS range        [3e-01, 9e+01]

Presolve removed 9 rows and 13 columns
Presolve time: 0.01s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    3.9648000e+03   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.01 seconds (0.00 work units)
Optimal objective  3.964800000e+03


In [12]:
# w/o uncertainty
vbasis = np.array(model_primal.getAttr(GRB.Attr.VBasis, model_primal.getVars()))
B0 = np.where(vbasis==0)[0]
N0 = np.setdiff1d(np.arange(model_primal.numVars), B0)
A = np.array(model_primal.getA().todense())
B = A[:, B0]
D = np.linalg.inv(B)
RHS = np.array(model_primal.getAttr(GRB.Attr.RHS))
xb= D @ RHS
dec_val_wo_uncertainty = np.zeros(A.shape[1])
dec_val_wo_uncertainty[B0] = xb
np.all(xb>=0), np.all(np.isclose(A @ dec_val_wo_uncertainty,RHS))

(np.True_, np.True_)

In [13]:
def notInfeasible(D, b1, tol=1e-9):
    b1_bar = np.array(D @ b1).ravel()
    return np.any(b1_bar < -tol)

In [15]:
# select the entering variable by ratio test; leaving by most negative
from helper import helper
# proba: use different omega scenario
dirpath = pathlib.Path('.\\pglib-opf-21.07\\')
case_names = [file.stem for file in dirpath.glob('*.m') if file.is_file()]
sorted_cases = sorted(case_names, key=lambda s: int(re.search(r'\d+', s).group()))
N_sample, percentage = 10, 0.05
sorted_cases = ['pglib_opf_case300_ieee']
iter = 0
for _ in range(N_sample):
    for case_name in sorted_cases:
        case_path = f'.\\pglib-opf-21.07\\{case_name}.m'
        case = CaseFrames(case_path)
        c, d0, fmax, nbranch, ngen, nbus, pmin, pmax = helper(case)
        model_primal = gp.Model(f"DC_Formulation_Model_primal_{case_name}")
        model_primal.params.OutputFlag = 0
        lpformulator_dc_body_primal(case, model_primal)
        omega_0 = create_scenario_multivariate(case, model_primal, 1, sigma_scaling=0)
        update_injection_constraints(case, model_primal, omega_0[0])
        model_primal.update()
        model_primal.optimize()
        if model_primal.Status != GRB.OPTIMAL:
            print('not optimal,', case_name)
            continue

        vbasis = np.array(model_primal.getAttr(GRB.Attr.VBasis, model_primal.getVars()))
        B0 = np.where(vbasis==0)[0]
        A = np.array(model_primal.getA().todense())
        B = A[:,B0]
        D = np.linalg.inv(B)
        leaving_enter_var_col = []
        obj_val_ratio = []
        omega_ratio = create_scenario_multivariate(case, model_primal, 1, sigma_scaling=percentage).ravel()
        RHS = np.array(model_primal.getAttr(GRB.Attr.RHS))
        b1_ratio = np.hstack([sum(d0 + omega_ratio) - pmin.sum(), 
                            pmax - pmin, 
                            fmax + case.M@(d0 + omega_ratio) - case.M@case.H@pmin, 
                            fmax - case.M@(d0 + omega_ratio) + case.M@case.H@pmin])
        while notInfeasible(D, b1_ratio):
            print(case_name)
            iter+=1
            b1_bar = np.array(D @ b1_ratio).ravel()
            cand_leaving_var = np.where(b1_bar<0)[0] # position within basis (subset of A)
            # use most negative b1_bar for leaving var, with argmin: the first occurence
            min_idx = np.argmin(b1_bar[cand_leaving_var])
            leaving_var_idx = cand_leaving_var[min_idx] # position within basis (subset of A)

            leaving_var_col = B0[leaving_var_idx] # position within original A
            N_idx = np.setdiff1d(arange(model_primal.NumVars-nbus), B0) # position within original A
            N_mat = A[:,N_idx]
            B_aj = np.array(D@N_mat) # matrix of non-basic variables in terms of basic variables
            row_nonbasic_cand_entering = B_aj[leaving_var_idx] # row of non-basic variables corresponding to leaving var
            cand_entering_var_in_A = N_idx[row_nonbasic_cand_entering<0] # index of nonbasic variables in original A with negative coeff in the row
            
            # select entering variable based on ratio test
            red_cost = c[cand_entering_var_in_A] - c[B0] @ D @ A[:, cand_entering_var_in_A] # reduced cost of nonbasic variables
            cand_entering_var_idx_in_N = np.where(np.isin(N_idx, cand_entering_var_in_A))[0] # position (index) of entering var candidates within N_idx
            ratio = -red_cost / B_aj[leaving_var_idx, cand_entering_var_idx_in_N] # ratio test by slicing B_aj within N_idx
            cand_entering_var = np.where(ratio == ratio.min())[0] # this index should return to cand_entering_var_idx_in_N
            min_cand_entering_var_idx_in_N = cand_entering_var_idx_in_N[cand_entering_var] # the return
            
            # if entering variable has multiple candidates with same minimum ratio, select randomly
            ent_var_idx_in_N = np.random.choice(min_cand_entering_var_idx_in_N)  # position (index) in N_idx
            pivot_val = B_aj[leaving_var_idx,ent_var_idx_in_N] # here was the problem
            new_pivot_row = (D[leaving_var_idx]/pivot_val).flatten()
            ent_var_col = N_idx[ent_var_idx_in_N] # from A original
            leaving_enter_var_col.append((leaving_var_col, ent_var_col))
            B0[leaving_var_idx] = ent_var_col 
            
            new_D_row_list = []
            D_old = D.copy() 
            pivot_col_tableau = B_aj[:, ent_var_idx_in_N].copy().ravel()

            for i in range(D_old.shape[0]):
                if i == leaving_var_idx:
                    new_D_row_list.append(new_pivot_row)
                else:
                    update_factor = pivot_col_tableau[i] # This is B_aj[i, ent_var_idx_in_N]
                    new_row = D_old[i] - (update_factor * new_pivot_row)
                    new_D_row_list.append(new_row)

            D = np.array(new_D_row_list)
            if np.all(D @ b1_ratio>=0):
                obj_val_ratio.append(c[B0] @ D @ b1_ratio)
    print('avg iter of 10:',iter/N_sample, case_name, percentage)        

FileNotFoundError: Can't find data at m:\projects\RF_DCOPF\pglib-opf-21.07\pglib_opf_case300_ieee.m

In [ ]:
# # # 0.05 10 times dantzig
# avg iter of 10: 0.0 pglib_opf_case3_lmbd 0.05
# avg iter of 10: 0.0 pglib_opf_case5_pjm 0.05
# avg iter of 10: 0.0 pglib_opf_case14_ieee 0.05
# avg iter of 10: 0.3 pglib_opf_case24_ieee_rts 0.05
# avg iter of 10: 0.0 pglib_opf_case30_as 0.05
# avg iter of 10: 0.0 pglib_opf_case30_ieee 0.05
# avg iter of 10: 0.1 pglib_opf_case39_epri 0.05
# avg iter of 10: 0.0 pglib_opf_case57_ieee 0.05
# avg iter of 10: 0.0 pglib_opf_case60_c 0.05
# avg iter of 10: 0.8 pglib_opf_case73_ieee_rts 0.05
# pglib_opf_case89_pegase ValueError: zero-size array to reduction operation minimum which has no identity
# avg iter of 10: 0.6 pglib_opf_case118_ieee 0.05
# avg iter of 10: 1.8 pglib_opf_case162_ieee_dtc 0.05
# avg iter of 10: 1.1 pglib_opf_case179_goc 0.05
# avg iter of 10: 0.0 pglib_opf_case200_activ 0.05
# avg iter of 10: 4.4 pglib_opf_case240_pserc 0.05
# avg iter of 10: 0.6 pglib_opf_case300_ieee 0.05

In [ ]:
# # 0.04 10 times dantzig
# avg iter of 10: 0.0 pglib_opf_case3_lmbd
# avg iter of 10: 0.0 pglib_opf_case5_pjm
# avg iter of 10: 0.0 pglib_opf_case14_ieee
# avg iter of 10: 0.2 pglib_opf_case24_ieee_rts
# avg iter of 10: 0.0 pglib_opf_case30_as
# avg iter of 10: 0.0 pglib_opf_case30_ieee
# avg iter of 10: 0.0 pglib_opf_case39_epri
# avg iter of 10: 0.0 pglib_opf_case57_ieee
# avg iter of 10: 0.0 pglib_opf_case60_c
# avg iter of 10: 0.9 pglib_opf_case73_ieee_rts
# pglib_opf_case89_pegase ValueError: zero-size array to reduction operation minimum which has no identity
# avg iter of 10: 0.7 pglib_opf_case118_ieee
# avg iter of 10: 1.6 pglib_opf_case162_ieee_dtc
# avg iter of 10: 0.6 pglib_opf_case179_goc
# avg iter of 10: 0.0 pglib_opf_case200_activ
# avg iter of 10: 8.4 pglib_opf_case240_pserc
# avg iter of 10: 0.2 pglib_opf_case300_ieee

In [131]:
# 0.03 10 times dantzig
# avg iter of 10: 0.0 pglib_opf_case3_lmbd
# avg iter of 10: 0.0 pglib_opf_case5_pjm
# avg iter of 10: 0.0 pglib_opf_case14_ieee
# avg iter of 10: 0.1 pglib_opf_case24_ieee_rts
# avg iter of 10: 0.0 pglib_opf_case30_as
# avg iter of 10: 0.0 pglib_opf_case30_ieee
# avg iter of 10: 0.1 pglib_opf_case39_epri
# avg iter of 10: 0.0 pglib_opf_case57_ieee
# avg iter of 10: 0.0 pglib_opf_case60_c
# avg iter of 10: 0.2 pglib_opf_case73_ieee_rts
# pglib_opf_case89_pegase ValueError: zero-size array to reduction operation minimum which has no identity
# avg iter of 10: 0.3 pglib_opf_case118_ieee
# avg iter of 10: 1.3 pglib_opf_case162_ieee_dtc
# avg iter of 10: 0.4 pglib_opf_case179_goc
# avg iter of 10: 0.0 pglib_opf_case200_activ
# avg iter of 10: 3.7 pglib_opf_case240_pserc
# avg iter of 10: 0.2 pglib_opf_case300_ieee
4648/60/24

3.227777777777778

In [ ]:
# # 0.05
# iter: 0 pglib_opf_case3_lmbd
# iter: 0 pglib_opf_case5_pjm
# iter: 0 pglib_opf_case14_ieee
# iter: 1 pglib_opf_case24_ieee_rts
# iter: 0 pglib_opf_case30_as
# iter: 0 pglib_opf_case30_ieee
# iter: 0 pglib_opf_case39_epri
# iter: 0 pglib_opf_case57_ieee
# iter: 0 pglib_opf_case60_c
# iter: 3 pglib_opf_case73_ieee_rts
# iter: 0 pglib_opf_case89_pegase
# iter: 0 pglib_opf_case118_ieee
# iter: 4 pglib_opf_case162_ieee_dtc
# iter: 1 pglib_opf_case179_goc
# iter: 0 pglib_opf_case200_activ
# iter: 20 pglib_opf_case240_pserc
# iter: 0 pglib_opf_case300_ieee

In [ ]:
# # 0.04
# iter: 0 pglib_opf_case3_lmbd
# iter: 0 pglib_opf_case5_pjm
# iter: 0 pglib_opf_case14_ieee
# iter: 0 pglib_opf_case24_ieee_rts
# iter: 0 pglib_opf_case30_as
# iter: 0 pglib_opf_case30_ieee
# iter: 0 pglib_opf_case39_epri
# iter: 0 pglib_opf_case57_ieee
# iter: 0 pglib_opf_case60_c
# iter: 2 pglib_opf_case73_ieee_rts
# pglib_opf_case89_pegase ValueError: zero-size array to reduction operation minimum which has no identity
# iter: 0 pglib_opf_case118_ieee
# iter: 3 pglib_opf_case162_ieee_dtc
# iter: 0 pglib_opf_case179_goc
# iter: 0 pglib_opf_case200_activ
# iter: 16 pglib_opf_case240_pserc
# iter: 0 pglib_opf_case300_ieee

In [ ]:
# 0.03
# iter: 0 pglib_opf_case3_lmbd
# iter: 0 pglib_opf_case5_pjm
# iter: 0 pglib_opf_case14_ieee
# iter: 0 pglib_opf_case24_ieee_rts
# iter: 0 pglib_opf_case30_as
# iter: 0 pglib_opf_case30_ieee
# iter: 0 pglib_opf_case39_epri
# iter: 0 pglib_opf_case57_ieee
# iter: 0 pglib_opf_case60_c
# iter: 1 pglib_opf_case73_ieee_rts
# iter: 0 pglib_opf_case89_pegase
# iter: 0 pglib_opf_case118_ieee
# iter: 2 pglib_opf_case162_ieee_dtc
# iter: 0 pglib_opf_case179_goc
# iter: 0 pglib_opf_case200_activ
# iter: 2 pglib_opf_case240_pserc
# iter: 1 pglib_opf_case300_ieee

In [ ]:
# solution feasibility check ratio
dec_val = D @ b1_ratio
N_idx = np.setdiff1d(arange(model_primal.NumVars), B0)
dec_val_all = np.zeros(model_primal.NumVars)
dec_val_all[B0] = dec_val
np.all(np.isclose(A @ dec_val_all, b1_ratio))

In [ ]:
# select randomly the leaving entering variable
vbasis = np.array(model_primal.getAttr(GRB.Attr.VBasis, model_primal.getVars()))
B0 = np.where(vbasis==0)[0]
A = np.array(model_primal.getA().todense())
B = A[:,B0]
D = np.linalg.inv(B)
iter = 0
leaving_enter_var_col = []
obj_val_prob = []
percentage = 0
percentage = 0.03
omega_prob = create_scenario_multivariate(case, model_primal, 1, sigma_scaling=percentage).ravel()
b1_prob = np.hstack([sum(d0 + omega_ratio) - pmin.sum(), 
                      pmax - pmin, 
                      fmax + case.M@(d0 + omega_ratio) - case.M@case.H@pmin, 
                      fmax - case.M@(d0 + omega_ratio) + case.M@case.H@pmin])
data_demand = np.load(f'..\\npz_{percentage}\\{case_name}_{percentage}_demand.npz')['arr_0']
bus_numbers = int(re.findall(r'\d+', case_name)[0])
bus_gen = case.gen.values[:,0][:,None]
ft_gen = np.broadcast_to(bus_gen, (bus_gen.shape[0],2))
ft_branch = case.branch.values[:,[0,1]]
ft_genbranch = np.concatenate([ft_gen,ft_gen,ft_branch,ft_branch])
Xsize = ft_genbranch.shape[0]
demand_arr = data_demand[:,:bus_numbers]
demand_broadcast = np.broadcast_to(demand_arr[:,None,:], (data_demand.shape[0], Xsize, bus_numbers))
ft_genbranch_broadcast = np.broadcast_to(ft_genbranch[None,:,:], (data_demand.shape[0],Xsize,ft_genbranch.shape[1]))

X_test = np.concatenate([ft_genbranch_broadcast, demand_broadcast], axis=2).reshape(data_demand.shape[0]*Xsize,-1)
y = data_demand[:,bus_numbers:].reshape(-1).astype(int)
d = d0+omega_prob
d = d[None, :]
d_test = np.concatenate([X_test[:,:2], np.repeat(d[:,:], [X_test.shape[0]], axis=0)], axis=1)
filename = f'..\\models_{percentage}\\model_LogisticRegression_{case_name}.joblib'
model_LogisticRegression = joblib.load(filename)
proba = model_LogisticRegression.predict_proba(d_test)    
while notInfeasible(D,b1_prob):
    iter+=1
    b1_bar = np.array(D @ b1_prob).ravel()
    cand_leaving_var = np.where(b1_bar<0)[0] # position within basis
    cand_leaving_var_proba = proba[cand_leaving_var,0] # 0 for non-basic (-1)
    # use maximum probability for leaving var (-1)
    leaving_proba_pos = np.where(cand_leaving_var_proba == cand_leaving_var_proba.max())[0] # position within cand_leaving_var_proba
    leaving_var_idx = cand_leaving_var[leaving_proba_pos] # position within basis
    # use the most negative b1_bar for leaving var; first occurence with argmin
    min_idx = np.argmin(b1_bar[leaving_var_idx])
    leaving_var_min = cand_leaving_var[min_idx]
    
    leaving_var_col = B0[leaving_var_min]
    N_idx = np.setdiff1d(arange(model_primal.NumVars), B0)
    N_mat = A[:,N_idx]
    B_aj = np.array(D@N_mat) 
    row_nonbasic_cand_entering = B_aj[leaving_var_idx].ravel() # row of non-basic variables corresponding to leaving var    
    cand_entering_var = N_idx[row_nonbasic_cand_entering<0] # index of nonbasic var with negative coeff in the row
    
    cand_entering_var_proba = proba[cand_entering_var,1] # 1 for basic (0)
    # use maximum probability for entering var
    entering_proba_pos = np.where(cand_entering_var_proba == cand_entering_var_proba.max())[0]
    entering_var_idx = cand_entering_var[entering_proba_pos] # index (from N) of nonbasic to enter
    # select entering variable based on ratio test among the maximum probabilities
    red_cost = c[entering_var_idx] - c[B0] @ D @ A[:, entering_var_idx] # reduced cost of nonbasic variables
    cand_entering_var_idx_in_N = np.where(np.isin(N_idx, entering_var_idx))[0] # position (index) of entering var candidates within N_idx
    ratio = -red_cost / B_aj[leaving_var_idx, cand_entering_var_idx_in_N] # ratio test by slicing B_aj within N_idx
    cand_entering_var = np.where(ratio == ratio.min())[0] # this index should return to cand_entering_var_idx_in_N
    min_cand_entering_var_idx_in_N = cand_entering_var_idx_in_N[cand_entering_var] # the return
    
    entering_var = N_idx[min_cand_entering_var_idx_in_N][0] # index of nonbasic to enter
    # entering_var = np.random.choice(entering_var_idx) # use this to select randomly among maximum probability candidates
    ent_var_idx = np.where(N_idx == entering_var)[0][0] # position of entering var in N_idx
    pivot_val = B_aj[leaving_var_idx, ent_var_idx][0]
    new_pivot_row = (D[leaving_var_idx]/pivot_val).flatten()

    leaving_enter_var_col.append((leaving_var_col, entering_var))
    B0[leaving_var_idx] = entering_var # update basis
    
    new_D_row_list = []
    D_old = D.copy() 
    pivot_col_tableau = B_aj[:, ent_var_idx].copy().ravel()
    for i in range(D_old.shape[0]):
        if i == leaving_var_idx:
            new_D_row_list.append(new_pivot_row)
        else:
            update_factor = pivot_col_tableau[i] # This is B_aj[i, ent_var_idx_in_N]
            new_row = D_old[i] - (update_factor * new_pivot_row)
            new_D_row_list.append(new_row)
    D = np.array(new_D_row_list)
    if np.all(D @ b1_prob>=0):
        obj_val_prob.append(c[B0] @ D @ b1_prob)

In [ ]:
# solution feasibility check proba
dec_val = D @ b1_prob
N_idx = np.setdiff1d(arange(model_primal.NumVars), B0)
dec_val_all = np.zeros(model_primal.NumVars)
dec_val_all[B0] = dec_val
np.all(np.isclose(A @ dec_val_all, b1_prob))

# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

In [ ]:
def notInfeasible(D, b1, tol=1e-9):
    b1_bar = np.array(D @ b1).ravel()
    return np.any(b1_bar < -tol)

In [129]:
from helper import helper
# proba: use different omega scenario
dirpath = pathlib.Path('.\\pglib-opf-21.07\\')
case_names = [file.stem for file in dirpath.glob('*.m') if file.is_file()]
sorted_cases = sorted(case_names, key=lambda s: int(re.search(r'\d+', s).group()))
N_sample, percentage = 10, 0.05
iter_dict = {}
obj_val_dict = {}
sorted_cases = ['pglib_opf_case300_ieee']
iter = 0
for nwk in range(N_sample):
    for case_name in sorted_cases:
        case_path = f'.\\pglib-opf-21.07\\{case_name}.m'
        case = CaseFrames(case_path)
        c, d0, fmax, nbranch, ngen, nbus, pmin, pmax = helper(case)
        model_primal = gp.Model(f"DC_Formulation_Model_primal_{case_name}")
        model_primal.params.OutputFlag = 0
        lpformulator_dc_body_primal(case, model_primal)
        omega_0 = create_scenario_multivariate(case, model_primal, 1, sigma_scaling=0)
        update_injection_constraints(case, model_primal, omega_0[0])
        model_primal.update()
        model_primal.optimize()
        if model_primal.Status != GRB.OPTIMAL:
            print('not optimal,', case_name)
            continue
        # select randomly the leaving entering variable
        vbasis = np.array(model_primal.getAttr(GRB.Attr.VBasis, model_primal.getVars()))
        B0 = np.where(vbasis==0)[0]
        A = np.array(model_primal.getA().todense())
        B = A[:,B0]
        D = np.linalg.inv(B)
        leaving_enter_var_col = []
        try:
            data_demand = np.load(f'..\\npz_{percentage}\\{case_name}_{percentage}_demand.npz')['arr_0']
        except:
            print('no data demand', case_name)
            continue
        bus_numbers = int(re.findall(r'\d+', case_name)[0])
        bus_gen = case.gen.values[:,0][:,None]
        ft_gen = np.broadcast_to(bus_gen, (bus_gen.shape[0],2))
        ft_branch = case.branch.values[:,[0,1]]
        ft_genbranch = np.concatenate([ft_gen,ft_gen,ft_branch,ft_branch])
        Xsize = ft_genbranch.shape[0]
        demand_arr = data_demand[:,:bus_numbers]
        demand_broadcast = np.broadcast_to(demand_arr[:,None,:], (data_demand.shape[0], Xsize, bus_numbers))
        ft_genbranch_broadcast = np.broadcast_to(ft_genbranch[None,:,:], (data_demand.shape[0],Xsize,ft_genbranch.shape[1]))
        X_test = np.concatenate([ft_genbranch_broadcast, demand_broadcast], axis=2).reshape(data_demand.shape[0]*Xsize,-1)
        y = data_demand[:,bus_numbers:].reshape(-1).astype(int)
        # X = data_demand[:,:,:-1]
        # X_test = X[0]
        omega_prob = create_scenario_multivariate(case, model_primal, 1, sigma_scaling=percentage).ravel()
        b1_prob = np.hstack([sum(d0 + omega_prob) - pmin.sum(), 
                        pmax - pmin, 
                        fmax + case.M@(d0 + omega_prob) - case.M@case.H@pmin, 
                        fmax - case.M@(d0 + omega_prob) + case.M@case.H@pmin])
        d = d0+omega_prob
        d = d[None, :]
        d_test = np.concatenate([X_test[:,:2], np.repeat(d[:,:], [X_test.shape[0]], axis=0)], axis=1)
        try:
            filename = f'..\\models_{percentage}\\model_LogisticRegression_{case_name}_{percentage}.joblib'
        except:
            print('no model', case_name)
            continue
        model_LogisticRegression = joblib.load(filename)
        proba = model_LogisticRegression.predict_proba(d_test)    
        # if np.all(D @ b1_prob>=0):
        #     obj_val_dict[case_name] = c[B0] @ D @ b1_prob
        #     iter_dict[case_name] = iter
        while notInfeasible(D,b1_prob, tol=1e-9):
            print(case_name)
            iter+=1
            b1_bar = np.array(D @ b1_prob).ravel()
            cand_leaving_var = np.where(b1_bar<0)[0] # position within basis
            cand_leaving_var_proba = proba[cand_leaving_var,0] # 0 for non-basic (-1)
            # use maximum probability for leaving var (-1)
            leaving_proba_pos = np.where(cand_leaving_var_proba == cand_leaving_var_proba.max())[0] # position within cand_leaving_var_proba
            leaving_var_idx = cand_leaving_var[leaving_proba_pos] # position within basis
            # use the most negative b1_bar for leaving var; first occurence with argmin
            min_idx = np.argmin(b1_bar[leaving_var_idx])
            leaving_var_min = cand_leaving_var[min_idx]
            
            leaving_var_col = B0[leaving_var_min]
            N_idx = np.setdiff1d(arange(model_primal.NumVars-nbus), B0)
            N_mat = A[:,N_idx]
            B_aj = np.array(D@N_mat) 
            row_nonbasic_cand_entering = B_aj[leaving_var_idx].ravel() # row of non-basic variables corresponding to leaving var    
            try:
                cand_entering_var = N_idx[row_nonbasic_cand_entering<0] # index of nonbasic var with negative coeff in the row
            except:
                print('gada cand_entering_var', case_name)
                break
            cand_entering_var_proba = proba[cand_entering_var,1] # wrong
            # use maximum probability for entering var
            try:
                entering_proba_pos = np.where(cand_entering_var_proba == cand_entering_var_proba.max())[0]
            except:
                print('no entering proba pos', case_name)
                break
            entering_var_idx = cand_entering_var[entering_proba_pos] # index (from N) of nonbasic to enter
            # select entering variable based on ratio test among the maximum probabilities
            red_cost = c[entering_var_idx] - c[B0] @ D @ A[:, entering_var_idx] # reduced cost of nonbasic variables
            cand_entering_var_idx_in_N = np.where(np.isin(N_idx, entering_var_idx))[0] # position (index) of entering var candidates within N_idx
            ratio = -red_cost / B_aj[leaving_var_idx, cand_entering_var_idx_in_N] # ratio test by slicing B_aj within N_idx
            cand_entering_var = np.where(ratio == ratio.min())[0] # this index should return to cand_entering_var_idx_in_N
            min_cand_entering_var_idx_in_N = cand_entering_var_idx_in_N[cand_entering_var] # the return
            
            entering_var = N_idx[min_cand_entering_var_idx_in_N][0] # index of nonbasic to enter
            # entering_var = np.random.choice(entering_var_idx) # use this to select randomly among maximum probability candidates
            ent_var_idx = np.where(N_idx == entering_var)[0][0] # position of entering var in N_idx
            pivot_val = B_aj[leaving_var_idx, ent_var_idx][0]
            new_pivot_row = (D[leaving_var_idx]/pivot_val).flatten()

            leaving_enter_var_col.append((leaving_var_col, entering_var))
            B0[leaving_var_idx] = entering_var # update basis
            
            new_D_row_list = []
            D_old = D.copy() 
            pivot_col_tableau = B_aj[:, ent_var_idx].copy().ravel()
            for i in range(D_old.shape[0]):
                if i == leaving_var_idx:
                    new_D_row_list.append(new_pivot_row)
                else:
                    update_factor = pivot_col_tableau[i] # This is B_aj[i, ent_var_idx_in_N]
                    new_row = D_old[i] - (update_factor * new_pivot_row)
                    new_D_row_list.append(new_row)
            D = np.array(new_D_row_list)
            if np.all(D @ b1_prob>=0):
                obj_val_dict[case_name] = c[B0] @ D @ b1_prob
                iter_dict[case_name] = iter
    print(f'avg iter of 10 (now {nwk}): {iter/N_sample} {case_name} {percentage}')

pglib_opf_case300_ieee
pglib_opf_case300_ieee
avg iter of 10 (now 0): 0.2 pglib_opf_case300_ieee 0.05
pglib_opf_case300_ieee
pglib_opf_case300_ieee
avg iter of 10 (now 1): 0.4 pglib_opf_case300_ieee 0.05
avg iter of 10 (now 2): 0.4 pglib_opf_case300_ieee 0.05
avg iter of 10 (now 3): 0.4 pglib_opf_case300_ieee 0.05
pglib_opf_case300_ieee
pglib_opf_case300_ieee
pglib_opf_case300_ieee
avg iter of 10 (now 4): 0.7 pglib_opf_case300_ieee 0.05
avg iter of 10 (now 5): 0.7 pglib_opf_case300_ieee 0.05
avg iter of 10 (now 6): 0.7 pglib_opf_case300_ieee 0.05
avg iter of 10 (now 7): 0.7 pglib_opf_case300_ieee 0.05
pglib_opf_case300_ieee
avg iter of 10 (now 8): 0.8 pglib_opf_case300_ieee 0.05
avg iter of 10 (now 9): 0.8 pglib_opf_case300_ieee 0.05


In [ ]:
# # # 0.05 10x proba 
# avg iter of 10: 0.0 pglib_opf_case3_lmbd 0.05
# avg iter of 10: 0.0 pglib_opf_case5_pjm 0.05
# avg iter of 10: 0.0 pglib_opf_case14_ieee 0.05
# avg iter of 10: 0.7 pglib_opf_case24_ieee_rts 0.05
# avg iter of 10: 0.0 pglib_opf_case30_as 0.05
# avg iter of 10: 0.0 pglib_opf_case30_ieee 0.05
# avg iter of 10: 0.1 pglib_opf_case39_epri 0.05
# avg iter of 10: 0.0 pglib_opf_case57_ieee 0.05
# avg iter of 10: 0.0 pglib_opf_case60_c 0.05
# avg iter of 10: 1.8 pglib_opf_case73_ieee_rts 0.05
# avg iter of 10: 0.8 pglib_opf_case89_pegase 0.05 no entering proba pos pglib_opf_case89_pegase
# avg iter of 10: 1.2 pglib_opf_case118_ieee 0.05
# avg iter of 10: 1.1 pglib_opf_case162_ieee_dtc 0.05
# avg iter of 10: 1.7 pglib_opf_case179_goc 0.05
# avg iter of 10: 0.0 pglib_opf_case200_activ 0.05
# avg iter of 10: 10.1 pglib_opf_case240_pserc 0.05 gada cand_entering_var pglib_opf_case240_pserc
# avg iter of 10 (now 9): 0.8 pglib_opf_case300_ieee 0.05

In [ ]:
# # 0.04 10x proba 
# avg iter of 10: 0.0 pglib_opf_case3_lmbd 0.04
# avg iter of 10: 0.0 pglib_opf_case5_pjm 0.04
# avg iter of 10: 0.0 pglib_opf_case14_ieee 0.04
# avg iter of 10: 0.7 pglib_opf_case24_ieee_rts 0.04
# avg iter of 10: 0.0 pglib_opf_case30_as 0.04
# avg iter of 10: 0.0 pglib_opf_case30_ieee 0.04
# avg iter of 10: 0.3 pglib_opf_case39_epri 0.04
# avg iter of 10: 0.0 pglib_opf_case57_ieee 0.04
# avg iter of 10: 0.0 pglib_opf_case60_c 0.04
# avg iter of 10: 1.7 pglib_opf_case73_ieee_rts 0.04
# avg iter of 10: 0.5 pglib_opf_case89_pegase 0.04 no entering proba pos pglib_opf_case89_pegase
# avg iter of 10: 0.4 pglib_opf_case118_ieee 0.04
# avg iter of 10: 1.3 pglib_opf_case162_ieee_dtc 0.04
# avg iter of 10: 1.0 pglib_opf_case179_goc 0.04
# avg iter of 10: 0.0 pglib_opf_case200_activ 0.04
# avg iter of 10: 13.5 pglib_opf_case240_pserc 0.04 gada cand_entering_var pglib_opf_case240_pserc
# avg iter of 10: 0.9 pglib_opf_case300_ieee 0.04


In [ ]:
# 0.03 10x proba 
# avg iter of 10: 0.0 pglib_opf_case3_lmbd 0.03
# avg iter of 10: 0.0 pglib_opf_case5_pjm 0.03
# avg iter of 10: 0.0 pglib_opf_case14_ieee 0.03
# avg iter of 10: 0.4 pglib_opf_case24_ieee_rts 0.03
# avg iter of 10: 0.0 pglib_opf_case30_as 0.03
# avg iter of 10: 0.0 pglib_opf_case30_ieee 0.03
# avg iter of 10: 0.1 pglib_opf_case39_epri 0.03
# avg iter of 10: 0.0 pglib_opf_case57_ieee 0.03
# avg iter of 10: 0.0 pglib_opf_case60_c 0.03
# avg iter of 10: 2.0 pglib_opf_case73_ieee_rts 0.03
# avg iter of 10: 0.6 pglib_opf_case89_pegase 0.03
# avg iter of 10: 0.6 pglib_opf_case118_ieee 0.03
# avg iter of 10: 1.0 pglib_opf_case162_ieee_dtc 0.03
# avg iter of 10: 0.6 pglib_opf_case179_goc 0.03
# avg iter of 10: 0.0 pglib_opf_case200_activ 0.03
# gada cand_entering_var pglib_opf_case240_pserc
# 0.02 gada cand_entering_var pglib_opf_case300_ieee

In [ ]:
# 0.05
iter_dict, obj_val_dict
# ({'pglib_opf_case3_lmbd': 0}, {'pglib_opf_case3_lmbd': 3858.204410525587})
# ({'pglib_opf_case5_pjm': 0}, {'pglib_opf_case5_pjm': 16130.031272450844})
# ({'pglib_opf_case14_ieee': 0}, {'pglib_opf_case14_ieee': 2071.909691248805})
# {'pglib_opf_case24_ieee_rts': 0}, {'pglib_opf_case24_ieee_rts': 28193.52495660982})
# ({'pglib_opf_case30_as': 0}, {'pglib_opf_case30_as': 430.44522233901})
# ({'pglib_opf_case30_ieee': 0}, {'pglib_opf_case30_ieee': 7814.859103636396})
# ({'pglib_opf_case39_epri': 0}, {'pglib_opf_case39_epri': 137682.47258608983})
# ({'pglib_opf_case57_ieee': 0}, {'pglib_opf_case57_ieee': 35664.64382413991})
# ({'pglib_opf_case60_c': 0}, {'pglib_opf_case60_c': 88187.90307801349})
# {'pglib_opf_case73_ieee_rts': 1}, {'pglib_opf_case73_ieee_rts': 78897.58126519449})
# {'pglib_opf_case89_pegase': 0}, {'pglib_opf_case89_pegase': 55854.380517641956})
# ({'pglib_opf_case118_ieee': 0}, {'pglib_opf_case118_ieee': 94102.43439342779})
# {'pglib_opf_case162_ieee_dtc': 1}, {'pglib_opf_case162_ieee_dtc': 97558.8708951372})
# ({'pglib_opf_case179_goc': 1}, {'pglib_opf_case179_goc': 61997.80922566508})
# {'pglib_opf_case200_activ': 0}, {'pglib_opf_case200_activ': 1736.6212747209804})
# ({'pglib_opf_case300_ieee': 0}, {'pglib_opf_case300_ieee': 517215.90291597776})

In [ ]:
# 0.04
# ({'pglib_opf_case3_lmbd': 0}, {'pglib_opf_case3_lmbd': 3894.5373655710096})
# ({'pglib_opf_case5_pjm': 0}, {'pglib_opf_case5_pjm': 17482.09960120362})
# ({'pglib_opf_case14_ieee': 0}, {'pglib_opf_case14_ieee': 2095.939353333673})
# ({'pglib_opf_case24_ieee_rts': 0}, {'pglib_opf_case24_ieee_rts': 25331.69355396618})
# ({'pglib_opf_case30_as': 0}, {'pglib_opf_case30_as': 413.96046703256746})
# ({'pglib_opf_case30_ieee': 0}, {'pglib_opf_case30_ieee': 7642.941882767169})
# ({'pglib_opf_case39_epri': 0}, {'pglib_opf_case39_epri': 134145.1517339807})
# ({'pglib_opf_case57_ieee': 0}, {'pglib_opf_case57_ieee': 34981.428737598086})
# ({'pglib_opf_case60_c': 0}, {'pglib_opf_case60_c': 87546.44522968208})
# ({'pglib_opf_case73_ieee_rts': 5}, {'pglib_opf_case73_ieee_rts': 85651.97953288157})
# ({'pglib_opf_case89_pegase': 0}, {'pglib_opf_case89_pegase': 58394.278707333666})
# ({'pglib_opf_case118_ieee': 0}, {'pglib_opf_case118_ieee': 93505.1838893654})
# ({'pglib_opf_case162_ieee_dtc': 1}, {'pglib_opf_case162_ieee_dtc': 103147.88241901674})
# ({'pglib_opf_case179_goc': 0}, {'pglib_opf_case179_goc': 48499.24875865586})
# {'pglib_opf_case200_activ': 0}, {'pglib_opf_case200_activ': 1507.7293844250826})
# ({'pglib_opf_case300_ieee': 2}, {'pglib_opf_case300_ieee': 516833.3666503376})


In [ ]:
# 0.03
# ({'pglib_opf_case3_lmbd': 0}, {'pglib_opf_case3_lmbd': 3933.8952463734054})
# ({'pglib_opf_case5_pjm': 0}, {'pglib_opf_case5_pjm': 17255.60696248834})
# ({'pglib_opf_case14_ieee': 0}, {'pglib_opf_case14_ieee': 2018.2367485708887})
# ({'pglib_opf_case24_ieee_rts': 0},{'pglib_opf_case24_ieee_rts': 27991.915611176664})
# ({'pglib_opf_case30_as': 0}, {'pglib_opf_case30_as': 424.7967919569067})
# ({'pglib_opf_case30_ieee': 0}, {'pglib_opf_case30_ieee': 7531.057512472738})
# ({'pglib_opf_case39_epri': 0}, {'pglib_opf_case39_epri': 138463.92959618574})
# ({'pglib_opf_case57_ieee': 0}, {'pglib_opf_case57_ieee': 34743.92180763582})
# ({'pglib_opf_case60_c': 0}, {'pglib_opf_case60_c': 85998.37044741103})
# ({'pglib_opf_case73_ieee_rts': 5}, {'pglib_opf_case73_ieee_rts': 85817.38398948473})
# ({'pglib_opf_case89_pegase': 1}, {'pglib_opf_case89_pegase': 54865.05702517321})
# ({'pglib_opf_case118_ieee': 0}, {'pglib_opf_case118_ieee': 93145.89820050506})
# ({'pglib_opf_case162_ieee_dtc': 0}, {'pglib_opf_case162_ieee_dtc': 101342.94934752208})
# ({'pglib_opf_case179_goc': 0}, {'pglib_opf_case179_goc': 43971.89485252354})
# ({'pglib_opf_case200_activ': 0}, {'pglib_opf_case200_activ': 1641.1567020191744})
# ({'pglib_opf_case300_ieee': 1}, {'pglib_opf_case300_ieee': 516987.95122137124})